In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings
from google.colab import drive
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

warnings.filterwarnings('ignore')
print("✅ All libraries imported successfully!")

In [ ]:
# Create sample dataset (works without internet)
np.random.seed(42)
n_samples = 614

data = {
    'Loan_ID': [f'LP{str(i).zfill(4)}' for i in range(n_samples)],
    'Gender': np.random.choice(['Male', 'Female'], n_samples, p=[0.8, 0.2]),
    'Married': np.random.choice(['Yes', 'No'], n_samples, p=[0.65, 0.35]),
    'Dependents': np.random.choice(['0', '1', '2', '3+'], n_samples, p=[0.5, 0.2, 0.2, 0.1]),
    'Education': np.random.choice(['Graduate', 'Not Graduate'], n_samples, p=[0.7, 0.3]),
    'Self_Employed': np.random.choice(['No', 'Yes'], n_samples, p=[0.85, 0.15]),
    'ApplicantIncome': np.random.randint(1500, 30000, n_samples),
    'CoapplicantIncome': np.random.randint(0, 20000, n_samples),
    'LoanAmount': np.random.randint(30, 700, n_samples),
    'Loan_Amount_Term': np.random.choice([360, 180, 120, 480], n_samples, p=[0.6, 0.2, 0.1, 0.1]),
    'Credit_History': np.random.choice([1.0, 0.0], n_samples, p=[0.8, 0.2]),
    'Property_Area': np.random.choice(['Urban', 'Rural', 'Semiurban'], n_samples, p=[0.4, 0.3, 0.3]),
    'Loan_Status': np.random.choice(['Y', 'N'], n_samples, p=[0.7, 0.3])
}
df = pd.DataFrame(data)
print(f"✅ Dataset created successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
print("="*50)
print("DATASET INFORMATION")
print("="*50)
print(df.info())

print("\n" + "="*50)
print("STATISTICAL SUMMARY")
print("="*50)
print(df.describe())

print("\n" + "="*50)
print("TARGET VARIABLE DISTRIBUTION")
print("="*50)
print(df['Loan_Status'].value_counts())
print(f"\nApproval Rate: {df['Loan_Status'].value_counts(normalize=True)['Y']*100:.2f}%")

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Loan Status Distribution
df['Loan_Status'].value_counts().plot(kind='bar', ax=axes[0,0], color=['green', 'red'])
axes[0,0].set_title('Loan Approval Distribution')
axes[0,0].set_xlabel('Loan Status (Y=Approved, N=Rejected)')
axes[0,0].set_ylabel('Count')

# 2. Gender Distribution
df['Gender'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%')
axes[0,1].set_title('Gender Distribution')

# 3. Education Distribution
df['Education'].value_counts().plot(kind='bar', ax=axes[0,2], color='skyblue')
axes[0,2].set_title('Education Distribution')
axes[0,2].set_xlabel('Education')
axes[0,2].set_ylabel('Count')

# 4. Property Area Distribution
df['Property_Area'].value_counts().plot(kind='bar', ax=axes[1,0], color='lightcoral')
axes[1,0].set_title('Property Area Distribution')
axes[1,0].set_xlabel('Property Area')
axes[1,0].set_ylabel('Count')

# 5. Credit History Distribution
credit_history_counts = df['Credit_History'].value_counts()
credit_history_counts.plot(kind='bar', ax=axes[1,1], color=['green', 'red'])
axes[1,1].set_title('Credit History Distribution (1=Good, 0=Bad)')
axes[1,1].set_xlabel('Credit History')
axes[1,1].set_ylabel('Count')

# 6. Self Employed Distribution
df['Self_Employed'].value_counts().plot(kind='pie', ax=axes[1,2], autopct='%1.1f%%')
axes[1,2].set_title('Self Employed Distribution')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Credit History vs Loan Approval
pd.crosstab(df['Credit_History'], df['Loan_Status']).plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Credit History vs Loan Approval')
axes[0,0].set_xlabel('Credit History (1=Good, 0=Bad)')
axes[0,0].set_ylabel('Count')
axes[0,0].legend(title='Loan Status')

# 2. Education vs Loan Approval
pd.crosstab(df['Education'], df['Loan_Status']).plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Education vs Loan Approval')
axes[0,1].set_xlabel('Education')
axes[0,1].set_ylabel('Count')

# 3. Property Area vs Loan Approval
pd.crosstab(df['Property_Area'], df['Loan_Status']).plot(kind='bar', ax=axes[0,2])
axes[0,2].set_title('Property Area vs Loan Approval')
axes[0,2].set_xlabel('Property Area')
axes[0,2].set_ylabel('Count')

# 4. Applicant Income Distribution
df.boxplot(column='ApplicantIncome', by='Loan_Status', ax=axes[1,0])
axes[1,0].set_title('Applicant Income vs Loan Approval')
axes[1,0].set_xlabel('Loan Status')
axes[1,0].set_ylabel('Income')

# 5. Loan Amount Distribution
df.boxplot(column='LoanAmount', by='Loan_Status', ax=axes[1,1])
axes[1,1].set_title('Loan Amount vs Loan Approval')
axes[1,1].set_xlabel('Loan Status')
axes[1,1].set_ylabel('Loan Amount')

# 6. Correlation Heatmap
numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
corr_df = df.copy()
corr_df['Loan_Status_Num'] = corr_df['Loan_Status'].map({'Y': 1, 'N': 0})
sns.heatmap(corr_df[numerical_cols + ['Loan_Status_Num']].corr(), annot=True, cmap='coolwarm', ax=axes[1,2])
axes[1,2].set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

In [ ]:
df_processed = df.copy()

print("Before preprocessing:")
print(f"Missing values: {df_processed.isnull().sum().sum()}\n")

# Handle missing values
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']
for col in categorical_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)

numerical_cols = ['LoanAmount', 'Loan_Amount_Term']
for col in numerical_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        df_processed[col].fillna(df_processed[col].median(), inplace=True)

print("After preprocessing:")
print(f"Missing values: {df_processed.isnull().sum().sum()}")
print("\n✅ Data preprocessing complete!")

In [ ]:
# Create new features
df_processed['TotalIncome'] = df_processed['ApplicantIncome'] + df_processed['CoapplicantIncome']
df_processed['LoanAmount'] = df_processed['LoanAmount'].replace(0, 1)
df_processed['IncomeLoanRatio'] = df_processed['TotalIncome'] / df_processed['LoanAmount']
df_processed['Dependents'] = df_processed['Dependents'].replace('3+', 3).astype(int)

print("New features created:")
print("✅ TotalIncome")
print("✅ IncomeLoanRatio")
print("✅ Dependents (converted to numeric)")

df_processed.head()

In [ ]:
label_encoders = {}

# Binary encoding
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed']
for col in binary_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# One-hot encoding for Property Area
df_processed = pd.get_dummies(df_processed, columns=['Property_Area'], drop_first=True)

# Target encoding
df_processed['Loan_Status'] = df_processed['Loan_Status'].map({'Y': 1, 'N': 0})

print("\n✅ Encoding complete!")
df_processed.head()

In [ ]:
# Select features
feature_columns = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
                   'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term',
                   'Credit_History', 'TotalIncome', 'IncomeLoanRatio',
                   'Property_Area_Semiurban', 'Property_Area_Urban']

# Filter to existing columns
feature_columns = [col for col in feature_columns if col in df_processed.columns]

# Separate features and target
X = df_processed[feature_columns]
y = df_processed['Loan_Status']

# Scale numerical features
scaler = StandardScaler()
numerical_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                      'Loan_Amount_Term', 'TotalIncome', 'IncomeLoanRatio']
available_numerical = [col for col in numerical_features if col in X.columns]
X[available_numerical] = scaler.fit_transform(X[available_numerical])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Features used: {len(feature_columns)}")
print(f"\n✅ Data ready for modeling!")

In [ ]:
# Initialize models (ALL required models from requirements)
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(random_state=42, probability=True)
}

results = {}
predictions = {}

print("="*60)
print("MODEL TRAINING AND EVALUATION")
print("="*60)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Model': model
    }
    predictions[name] = y_pred

    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

# Cross-validation for Random Forest
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
cv_scores = cross_val_score(rf_model, X, y, cv=5)
print(f"\n{'='*60}")
print(f"Random Forest - 5-Fold Cross Validation:")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

In [ ]:
# Find best model
best_model_name = max(results, key=lambda x: results[x]['Accuracy'])
best_model = results[best_model_name]['Model']

print(f"🏆 BEST MODEL: {best_model_name}")
print(f"Accuracy: {results[best_model_name]['Accuracy']:.4f}\n")

# Plot confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, (name, _) in enumerate(results.items()):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Rejected', 'Approved'],
                yticklabels=['Rejected', 'Approved'])
    axes[idx].set_title(f'{name}\nAccuracy: {results[name]["Accuracy"]:.3f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for idx in range(len(results), 6):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Classification report
print("\n" + "="*60)
print(f"CLASSIFICATION REPORT - {best_model_name}")
print("="*60)
print(classification_report(y_test, predictions[best_model_name],
                           target_names=['Rejected (0)', 'Approved (1)']))

In [ ]:
# Feature importance from Random Forest
rf_model = results['Random Forest']['Model']
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Most Important Features for Loan Approval')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:")
for idx, row in feature_importance.head().iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))

for name, model_info in results.items():
    model = model_info['Model']
    if hasattr(model, "predict_proba"):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("Performing Grid Search for Random Forest Optimization...")
print("-"*50)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("\n" + "="*50)
print("BEST PARAMETERS FOUND:")
print("="*50)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Model Test Accuracy: {tuned_accuracy:.4f}")

original_accuracy = results['Random Forest']['Accuracy']
print(f"\nImprovement: {(tuned_accuracy - original_accuracy)*100:.2f}%")

In [ ]:
def predict_loan_approval(applicant_data, model, scaler, feature_columns, label_encoders):
    """Predict loan approval for a single applicant"""

    input_df = pd.DataFrame([applicant_data])

    # Encode categorical variables
    for col, le in label_encoders.items():
        if col in input_df.columns:
            input_df[col] = le.transform(input_df[col].astype(str))

    # Create new features
    input_df['TotalIncome'] = input_df['ApplicantIncome'] + input_df['CoapplicantIncome']
    input_df['IncomeLoanRatio'] = input_df['TotalIncome'] / input_df['LoanAmount']

    # One-hot encode Property Area
    if 'Property_Area' in input_df.columns:
        input_df = pd.get_dummies(input_df, columns=['Property_Area'], drop_first=True)

    # Ensure all columns exist
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0

    # Select and scale features
    available_features = [col for col in feature_columns if col in input_df.columns]
    input_features = input_df[available_features]

    numerical_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                          'Loan_Amount_Term', 'TotalIncome', 'IncomeLoanRatio']
    available_numerical = [col for col in numerical_features if col in input_features.columns]
    if available_numerical:
        input_features[available_numerical] = scaler.transform(input_features[available_numerical])

    # Make prediction
    prediction = model.predict(input_features)[0]
    probability = model.predict_proba(input_features)[0]

    return prediction, probability

# Test prediction
sample_applicant = {
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': 2,
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 5000,
    'CoapplicantIncome': 2000,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1,
    'Property_Area': 'Urban'
}

prediction, probability = predict_loan_approval(
    sample_applicant, best_rf, scaler, feature_columns, label_encoders
)

print("="*50)
print("LOAN APPROVAL PREDICTION RESULT")
print("="*50)
print(f"\nPrediction: {'✅ APPROVED' if prediction == 1 else '❌ REJECTED'}")
print(f"Confidence: {probability[prediction]*100:.2f}%")
print(f"Risk Assessment: {'Low Risk' if prediction == 1 else 'High Risk'}")

In [ ]:
# Save model - FIXED version with no errors
import os
import joblib

# Save model locally (always works)
model_artifacts = {
    'model': best_rf,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'feature_columns': feature_columns,
    'model_name': best_model_name
}

# Save locally
joblib.dump(model_artifacts, 'loan_approval_model.pkl')
print("✅ Model saved locally as 'loan_approval_model.pkl'")

# Try to save to Google Drive (if mounted)
try:
    if os.path.exists('/content/drive'):
        joblib.dump(model_artifacts, '/content/drive/MyDrive/loan_approval_model.pkl')
        print("✅ Model also saved to Google Drive")
    else:
        print("⚠️ Google Drive not mounted - skipping Drive save")
except Exception as e:
    print(f"⚠️ Could not save to Drive: {e}")

# Download to computer
from google.colab import files
try:
    files.download('loan_approval_model.pkl')
    print("✅ Model downloaded to your computer!")
except:
    print("⚠️ Could not auto-download. Use Files panel to download manually.")

print("\n📦 To load the model later:")
print("  import joblib")
print("  artifacts = joblib.load('loan_approval_model.pkl')")
print("  model = artifacts['model']")

In [ ]:
fig = plt.figure(figsize=(20, 12))

# 1. Model Comparison
ax1 = fig.add_subplot(2, 3, 1)
models_names = list(results.keys())
accuracies = [results[m]['Accuracy'] for m in models_names]
bars = ax1.bar(models_names, accuracies, color='skyblue')
ax1.set_title('Model Accuracy Comparison')
ax1.set_ylabel('Accuracy')
ax1.set_ylim([0, 1])
ax1.tick_params(axis='x', rotation=45)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom')

# 2. Confusion Matrix for Best Model
ax2 = fig.add_subplot(2, 3, 2)
cm = confusion_matrix(y_test, predictions[best_model_name])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2)
ax2.set_title(f'Confusion Matrix - {best_model_name}')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')

# 3. Precision-Recall Comparison
ax3 = fig.add_subplot(2, 3, 3)
x = np.arange(len(models_names))
width = 0.25
precision_scores = [results[m]['Precision'] for m in models_names]
recall_scores = [results[m]['Recall'] for m in models_names]
ax3.bar(x - width/2, precision_scores, width, label='Precision', color='green')
ax3.bar(x + width/2, recall_scores, width, label='Recall', color='orange')
ax3.set_xticks(x)
ax3.set_xticklabels(models_names, rotation=45, ha='right')
ax3.set_title('Precision vs Recall by Model')
ax3.legend()
ax3.set_ylim([0, 1])

# 4. Feature Importance
ax4 = fig.add_subplot(2, 3, 4)
top_features = feature_importance.head(8)
ax4.barh(top_features['Feature'], top_features['Importance'], color='coral')
ax4.set_title('Top 8 Feature Importance')
ax4.set_xlabel('Importance')

# 5. Model Performance Heatmap
ax5 = fig.add_subplot(2, 3, 5)
performance_data = []
for name in models_names:
    performance_data.append([
        results[name]['Accuracy'],
        results[name]['Precision'],
        results[name]['Recall'],
        results[name]['F1-Score']
    ])
sns.heatmap(performance_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax5,
            xticklabels=['Acc', 'Prec', 'Rec', 'F1'],
            yticklabels=models_names)
ax5.set_title('Model Performance Heatmap')

# 6. Summary Table
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('tight')
ax6.axis('off')
summary_data = [
    ['Best Model', best_model_name],
    ['Test Accuracy', f"{results[best_model_name]['Accuracy']:.4f}"],
    ['Precision', f"{results[best_model_name]['Precision']:.4f}"],
    ['Recall', f"{results[best_model_name]['Recall']:.4f}"],
    ['F1-Score', f"{results[best_model_name]['F1-Score']:.4f}"],
    ['Training Samples', len(X_train)],
    ['Testing Samples', len(X_test)],
    ['Features Used', len(feature_columns)]
]
table = ax6.table(cellText=summary_data, colLabels=['Metric', 'Value'],
                  cellLoc='left', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

plt.suptitle('LOAN APPROVAL PREDICTION SYSTEM - FINAL REPORT', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("PROJECT SUMMARY")
print("="*60)
print(f"✅ Best Performing Model: {best_model_name}")
print(f"✅ Overall Accuracy: {results[best_model_name]['Accuracy']*100:.2f}%")
print(f"✅ Precision: {results[best_model_name]['Precision']*100:.2f}%")
print(f"✅ Recall: {results[best_model_name]['Recall']*100:.2f}%")
print(f"✅ F1-Score: {results[best_model_name]['F1-Score']*100:.2f}%")

In [ ]:
# Quick interactive prediction
print("\n" + "="*50)
print("QUICK PREDICTION TEST")
print("="*50)

test_cases = [
    {'Gender':'Male', 'Married':'Yes', 'Dependents':2, 'Education':'Graduate',
     'Self_Employed':'No', 'ApplicantIncome':8000, 'CoapplicantIncome':3000,
     'LoanAmount':200, 'Loan_Amount_Term':360, 'Credit_History':1, 'Property_Area':'Urban'},

    {'Gender':'Female', 'Married':'No', 'Dependents':0, 'Education':'Not Graduate',
     'Self_Employed':'Yes', 'ApplicantIncome':2000, 'CoapplicantIncome':0,
     'LoanAmount':300, 'Loan_Amount_Term':180, 'Credit_History':0, 'Property_Area':'Rural'}
]

for i, applicant in enumerate(test_cases, 1):
    pred, prob = predict_loan_approval(applicant, best_rf, scaler, feature_columns, label_encoders)
    print(f"\nTest Case {i}:")
    print(f"  Income: ${applicant['ApplicantIncome']}, Credit: {applicant['Credit_History']}")
    print(f"  Result: {'✅ APPROVED' if pred == 1 else '❌ REJECTED'} (Confidence: {prob[pred]*100:.1f}%)")

In [ ]:
# ============================================================
# LOAN APPROVAL PREDICTION SYSTEM - COMPLETE FIXED UI
# FULLY VISIBLE DROPDOWNS & SLIDERS - NO CUT-OFF
# ============================================================

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd
import numpy as np
from ipywidgets import Layout, VBox, HBox
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STYLING - FULL VISIBILITY WITH PROPER SPACING
# ============================================================
display(HTML("""
<style>
/* Global background - Light purple */
body, .p-Widget, .widget-area {
    background: linear-gradient(135deg, #f5f0ff 0%, #e8dfff 100%) !important;
}

/* Main container */
.main-container {
    max-width: 1400px;
    margin: 0 auto;
    padding: 20px;
    font-family: 'Segoe UI', Arial, sans-serif;
}

/* Header */
.professional-header {
    background: linear-gradient(135deg, #1a237e 0%, #283593 50%, #1a237e 100%);
    padding: 30px;
    border-radius: 20px;
    text-align: center;
    margin-bottom: 30px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.2);
}
.professional-header h1 {
    color: white !important;
    font-size: 38px;
    margin: 0;
}
.professional-header p {
    color: #e8eaf6 !important;
    font-size: 16px;
    margin: 10px 0 0 0;
}
.accuracy-badge {
    background: #ffd700;
    color: #1a237e !important;
    display: inline-block;
    padding: 8px 25px;
    border-radius: 30px;
    margin-top: 15px;
    font-weight: bold;
    font-size: 16px;
}

/* Cards - Wider and more padding */
.professional-card {
    background: white !important;
    border-radius: 16px !important;
    padding: 25px !important;
    margin: 12px !important;
    box-shadow: 0 4px 20px rgba(0,0,0,0.1) !important;
    border: 1px solid #d4b8ff !important;
    min-width: 320px !important;
}

/* Section titles */
.section-title {
    background: linear-gradient(135deg, #1a237e 0%, #283593 100%) !important;
    color: white !important;
    padding: 12px 15px !important;
    border-radius: 12px !important;
    margin: -25px -25px 20px -25px !important;
    font-size: 18px !important;
    font-weight: bold !important;
    text-align: center !important;
}

/* Labels - BLACK with proper spacing */
label, .widget-label, .description {
    color: #000000 !important;
    font-weight: bold !important;
    font-size: 14px !important;
    margin-bottom: 8px !important;
    display: block !important;
}

/* Input fields - FULL WIDTH with proper height */
input, select, textarea, .widget-dropdown select, .widget-int-text input {
    color: #000000 !important;
    background-color: #ffffff !important;
    border: 2px solid #1a237e !important;
    border-radius: 8px !important;
    padding: 10px 12px !important;
    font-size: 14px !important;
    width: 100% !important;
    min-height: 40px !important;
}

/* Dropdown - FIXED: Full visibility */
.widget-dropdown {
    width: 100% !important;
    min-width: 280px !important;
}
.widget-dropdown select {
    width: 100% !important;
    min-height: 42px !important;
    overflow: visible !important;
}
.widget-dropdown select option {
    padding: 10px !important;
    min-height: 35px !important;
}

/* Radio buttons - Better spacing */
.widget-radio {
    margin: 10px 0 !important;
    width: 100% !important;
}
.widget-radio label {
    color: #000000 !important;
    font-weight: 500 !important;
    margin-left: 8px !important;
}
.widget-radio input {
    accent-color: #1a237e !important;
    width: 16px !important;
    height: 16px !important;
}

/* Sliders - FIXED: Full width with proper padding */
.widget-slider {
    margin: 15px 0 25px 0 !important;
    width: 100% !important;
}
.noUi-target {
    width: 100% !important;
    height: 8px !important;
}
.noUi-connect {
    background: #1a237e !important;
}
.noUi-handle {
    width: 20px !important;
    height: 20px !important;
    border-radius: 50% !important;
}

/* Value displays */
.value-display {
    background: #e8eaf6 !important;
    padding: 8px 12px !important;
    border-radius: 8px !important;
    margin: 8px 0 !important;
    text-align: center !important;
    border: 1px solid #1a237e !important;
    font-weight: bold !important;
    color: #1a237e !important;
}

/* Buttons */
.btn-predict {
    background: linear-gradient(135deg, #28a745 0%, #20c997 100%) !important;
    color: white !important;
    font-size: 18px !important;
    font-weight: bold !important;
    border: none !important;
    border-radius: 50px !important;
    padding: 12px 30px !important;
    cursor: pointer !important;
}
.btn-clear {
    background: linear-gradient(135deg, #dc3545 0%, #c82333 100%) !important;
    color: white !important;
    font-size: 16px !important;
    font-weight: bold !important;
    border: none !important;
    border-radius: 50px !important;
    padding: 12px 25px !important;
    cursor: pointer !important;
}
.btn-reset {
    background: linear-gradient(135deg, #ffc107 0%, #ff8c00 100%) !important;
    color: black !important;
    font-size: 16px !important;
    font-weight: bold !important;
    border: none !important;
    border-radius: 50px !important;
    padding: 12px 25px !important;
    cursor: pointer !important;
}

/* Output container */
.output-container {
    background: white !important;
    border-radius: 16px !important;
    padding: 25px !important;
    margin-top: 25px !important;
    border: 2px solid #1a237e !important;
}

/* Result cards */
.approved-card {
    background: #d4edda !important;
    border: 3px solid #28a745 !important;
    border-radius: 16px !important;
    padding: 25px !important;
    margin: 15px 0 !important;
    text-align: center;
}
.rejected-card {
    background: #f8d7da !important;
    border: 3px solid #dc3545 !important;
    border-radius: 16px !important;
    padding: 25px !important;
    margin: 15px 0 !important;
    text-align: center;
}
.approved-text {
    color: #155724 !important;
    font-size: 32px !important;
    font-weight: bold !important;
}
.rejected-text {
    color: #721c24 !important;
    font-size: 32px !important;
    font-weight: bold !important;
}

/* Progress bar */
.progress-wrapper {
    background: #e0e0e0 !important;
    border-radius: 30px !important;
    overflow: hidden !important;
    height: 40px !important;
    margin: 15px 0 !important;
}
.progress-fill {
    height: 40px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    color: white !important;
    font-weight: bold !important;
    font-size: 16px !important;
}

/* Tables */
.data-table {
    width: 100%;
    border-collapse: collapse;
    margin: 15px 0;
    background: white;
    border-radius: 12px;
    overflow: auto;
}
.data-table td {
    padding: 12px 15px;
    border: 1px solid #ddd;
    color: #000000 !important;
}
.data-table tr:nth-child(even) {
    background-color: #f9f9f9;
}
.data-table td:first-child {
    font-weight: bold;
    background-color: #f0f0f0;
    width: 35%;
}

/* Risk boxes */
.risk-box {
    padding: 20px !important;
    border-radius: 12px !important;
    margin: 15px 0 !important;
}
.risk-box-low {
    background: #d4edda !important;
    border: 2px solid #28a745 !important;
}
.risk-box-medium {
    background: #fff3cd !important;
    border: 2px solid #ffc107 !important;
}
.risk-box-high {
    background: #f8d7da !important;
    border: 2px solid #dc3545 !important;
}

/* Tips panel */
.tips-panel {
    background: linear-gradient(135deg, #1a237e 0%, #283593 100%) !important;
    border-radius: 16px !important;
    padding: 20px !important;
    margin-top: 25px !important;
}
.tips-panel h4 {
    color: #ffd700 !important;
    margin: 0 0 15px 0 !important;
    font-size: 18px !important;
}
.tips-panel li {
    color: white !important;
    margin: 10px 0 !important;
}

/* Three column layout - FIXED: Better spacing */
.three-columns {
    display: flex;
    flex-wrap: wrap;
    gap: 20px;
    margin-bottom: 20px;
}
.column {
    flex: 1;
    min-width: 330px;
    max-width: 100%;
}

/* Button row */
.button-row {
    display: flex;
    justify-content: center;
    gap: 20px;
    margin: 25px 0;
    flex-wrap: wrap;
}

/* Fix for cut-off content */
.widget-vbox {
    width: 100% !important;
}
.widget-hbox {
    width: 100% !important;
}
</style>
"""))

# ============================================================
# CREATE INPUT WIDGETS - WITH PROPER SIZING
# ============================================================

# Column 1 - Personal Information
gender = widgets.RadioButtons(
    options=['Male', 'Female'],
    value='Male',
    description='Gender:',
    style={'description_width': '100px'},
    layout=Layout(width='100%', min_width='280px')
)

married = widgets.RadioButtons(
    options=['Yes', 'No'],
    value='Yes',
    description='Married:',
    style={'description_width': '100px'},
    layout=Layout(width='100%', min_width='280px')
)

dependents = widgets.Dropdown(
    options=[0, 1, 2, 3, 4, 5],
    value=0,
    description='Dependents:',
    style={'description_width': '100px'},
    layout=Layout(width='100%', min_width='280px')
)

education = widgets.RadioButtons(
    options=['Graduate', 'Not Graduate'],
    value='Graduate',
    description='Education:',
    style={'description_width': '100px'},
    layout=Layout(width='100%', min_width='280px')
)

self_employed = widgets.RadioButtons(
    options=['No', 'Yes'],
    value='No',
    description='Self Employed:',
    style={'description_width': '100px'},
    layout=Layout(width='100%', min_width='280px')
)

# Column 2 - Financial Information (FIXED: Sliders now fully visible)
applicant_income = widgets.IntSlider(
    value=5000,
    min=0,
    max=50000,
    step=500,
    description='Applicant Income ($):',
    style={'description_width': '150px'},
    layout=Layout(width='100%', min_width='280px')
)
applicant_value = widgets.HTML("<div class='value-display'>💰 Current: $5,000</div>")

def update_app(change):
    applicant_value.value = f"<div class='value-display'>💰 Current: ${change['new']:,}</div>"
applicant_income.observe(update_app, 'value')

coapplicant_income = widgets.IntSlider(
    value=0,
    min=0,
    max=50000,
    step=500,
    description='Co-applicant Income ($):',
    style={'description_width': '150px'},
    layout=Layout(width='100%', min_width='280px')
)
coapp_value = widgets.HTML("<div class='value-display'>👥 Current: $0</div>")

def update_coapp(change):
    coapp_value.value = f"<div class='value-display'>👥 Current: ${change['new']:,}</div>"
coapplicant_income.observe(update_coapp, 'value')

total_display = widgets.HTML("<div style='background:#e8eaf6; padding:12px; border-radius:10px; text-align:center; margin:10px 0; border:2px solid #1a237e'><b style='color:#1a237e; font-size:18px'>💰 TOTAL INCOME: $5,000</b></div>")

def update_total(change):
    total = applicant_income.value + coapplicant_income.value
    total_display.value = f"<div style='background:#e8eaf6; padding:12px; border-radius:10px; text-align:center; margin:10px 0; border:2px solid #1a237e'><b style='color:#1a237e; font-size:18px'>💰 TOTAL INCOME: ${total:,}</b></div>"
applicant_income.observe(update_total, 'value')
coapplicant_income.observe(update_total, 'value')

loan_amount = widgets.IntSlider(
    value=150,
    min=0,
    max=1000,
    step=10,
    description='Loan Amount ($K):',
    style={'description_width': '150px'},
    layout=Layout(width='100%', min_width='280px')
)
loan_value = widgets.HTML("<div class='value-display'>🏦 Current: $150K</div>")

def update_loan(change):
    loan_value.value = f"<div class='value-display'>🏦 Current: ${change['new']}K</div>"
loan_amount.observe(update_loan, 'value')

loan_term = widgets.Dropdown(
    options=[120, 180, 240, 300, 360, 480],
    value=360,
    description='Loan Term (months):',
    style={'description_width': '150px'},
    layout=Layout(width='100%', min_width='280px')
)

# Column 3 - Credit Information
credit_history = widgets.RadioButtons(
    options=['Good (1)', 'Bad (0)'],
    value='Good (1)',
    description='Credit History:',
    style={'description_width': '120px'},
    layout=Layout(width='100%', min_width='280px')
)

property_area = widgets.RadioButtons(
    options=['Urban', 'Semiurban', 'Rural'],
    value='Urban',
    description='Property Area:',
    style={'description_width': '120px'},
    layout=Layout(width='100%', min_width='280px')
)

# ============================================================
# BUTTONS
# ============================================================
predict_btn = widgets.Button(
    description='🔮 PREDICT LOAN STATUS',
    layout=Layout(width='260px', height='55px')
)
predict_btn.add_class('btn-predict')

clear_btn = widgets.Button(
    description='🗑️ CLEAR FORM',
    layout=Layout(width='200px', height='55px')
)
clear_btn.add_class('btn-clear')

reset_btn = widgets.Button(
    description='🔄 RESET',
    layout=Layout(width='180px', height='55px')
)
reset_btn.add_class('btn-reset')

# ============================================================
# OUTPUT AREA
# ============================================================
output = widgets.Output(layout=Layout(
    width='100%',
    min_height='450px',
    background='white',
    border='2px solid #1a237e',
    border_radius='16px',
    padding='20px'
))

# ============================================================
# PREDICTION FUNCTIONS
# ============================================================
def create_progress(percentage, color):
    return f"""
    <div class="progress-wrapper">
        <div class="progress-fill" style="width: {percentage}%; background: {color};">
            {percentage:.1f}%
        </div>
    </div>
    """

def on_predict(b):
    with output:
        clear_output(wait=True)

        credit_val = 1 if credit_history.value == 'Good (1)' else 0

        applicant_data = {
            'Gender': gender.value,
            'Married': married.value,
            'Dependents': dependents.value,
            'Education': education.value,
            'Self_Employed': self_employed.value,
            'ApplicantIncome': applicant_income.value,
            'CoapplicantIncome': coapplicant_income.value,
            'LoanAmount': loan_amount.value,
            'Loan_Amount_Term': loan_term.value,
            'Credit_History': credit_val,
            'Property_Area': property_area.value,
            'TotalIncome': applicant_income.value + coapplicant_income.value
        }

        display(HTML("<div style='text-align:center; padding:30px; background:#e8eaf6; border-radius:12px'><h3 style='color:#1a237e'>🔄 Analyzing Application...</h3></div>"))

        try:
            prediction, probability = predict_loan_approval(applicant_data, best_rf, scaler, feature_columns, label_encoders)
            confidence = probability[prediction] * 100

            clear_output(wait=True)

            # Result Card
            if prediction == 1:
                display(HTML(f"""
                <div class="approved-card">
                    <div class="approved-text">✅ LOAN APPROVED</div>
                    <div style="color:#155724; font-size:18px; margin:10px 0">Confidence Level: <strong>{confidence:.1f}%</strong></div>
                    {create_progress(confidence, 'linear-gradient(135deg, #28a745, #20c997)')}
                </div>
                """))
            else:
                display(HTML(f"""
                <div class="rejected-card">
                    <div class="rejected-text">❌ LOAN REJECTED</div>
                    <div style="color:#721c24; font-size:18px; margin:10px 0">Confidence Level: <strong>{confidence:.1f}%</strong></div>
                    {create_progress(confidence, 'linear-gradient(135deg, #dc3545, #c82333)')}
                </div>
                """))

            # Application Summary Table
            display(HTML("<h3 style='color:#1a237e; margin: 20px 0 10px 0'>📋 Application Summary</h3>"))
            display(HTML(f"""
            <table class="data-table">
                <tr><td><b>Gender</b></td><td>{gender.value}</td><td><b>Marital Status</b></td><td>{married.value}</td></tr>
                <tr><td><b>Dependents</b></td><td>{dependents.value}</td><td><b>Education</b></td><td>{education.value}</td></tr>
                <tr><td><b>Self Employed</b></td><td>{self_employed.value}</td><td><b>Credit History</b></td><td>{credit_history.value}</td></tr>
                <tr><td><b>Applicant Income</b></td><td>${applicant_income.value:,}</td><td><b>Co-applicant Income</b></td><td>${coapplicant_income.value:,}</td></tr>
                <tr><td><b>Total Income</b></td><td>${applicant_income.value + coapplicant_income.value:,}</td><td><b>Loan Amount</b></td><td>${loan_amount.value}K</td></tr>
                <tr><td><b>Loan Term</b></td><td>{loan_term.value} months</td><td><b>Property Area</b></td><td>{property_area.value}</td></tr>
            </table>
            """))

            # Calculate Risk Score
            risk = 50
            if credit_val == 0:
                risk += 30
            else:
                risk -= 15
            if applicant_income.value < 3000:
                risk += 25
            elif applicant_income.value > 10000:
                risk -= 10
            if loan_amount.value > 500:
                risk += 20
            elif loan_amount.value < 150:
                risk -= 10
            if dependents.value > 3:
                risk += 10

            risk = min(100, max(0, risk))
            if prediction == 1:
                risk = max(0, risk - 15)
            else:
                risk = min(100, risk + 15)

            # Risk level
            if risk < 30:
                risk_class = "risk-box-low"
                risk_level = "🟢 LOW RISK"
                risk_color = "#28a745"
            elif risk < 60:
                risk_class = "risk-box-medium"
                risk_level = "🟡 MEDIUM RISK"
                risk_color = "#ffc107"
            else:
                risk_class = "risk-box-high"
                risk_level = "🔴 HIGH RISK"
                risk_color = "#dc3545"

            # RISK ASSESSMENT - HIGHLY VISIBLE
            display(HTML(f"""
            <h3 style='color:#1a237e; margin: 25px 0 10px 0; font-size:20px'>📊 RISK ASSESSMENT</h3>
            <div class="{risk_class}" style='padding: 20px; border-radius: 12px; text-align: center;'>
                <div style='font-size: 24px; font-weight: bold; color: {risk_color};'>{risk_level}</div>
                <div style='font-size: 20px; margin: 10px 0; color: #000000;'>
                    <strong>Risk Score: {risk:.1f}%</strong>
                </div>
            </div>
            """))

            if prediction == 0:
                display(HTML("<h3 style='color:#1a237e; margin: 20px 0 10px 0'>💡 Recommendations for Approval</h3>"))
                if credit_val == 0:
                    display(HTML("<div style='background:#fff3cd; padding:10px; margin:5px 0; border-radius:8px'>📌 Improve your credit history by paying bills on time</div>"))
                if applicant_income.value < 3000:
                    display(HTML("<div style='background:#fff3cd; padding:10px; margin:5px 0; border-radius:8px'>📌 Increase your income or provide additional income proof</div>"))
                if loan_amount.value > 500:
                    display(HTML("<div style='background:#fff3cd; padding:10px; margin:5px 0; border-radius:8px'>📌 Request a smaller loan amount or provide larger down payment</div>"))
                if coapplicant_income.value == 0:
                    display(HTML("<div style='background:#fff3cd; padding:10px; margin:5px 0; border-radius:8px'>📌 Consider adding a co-applicant with good credit history</div>"))

        except Exception as e:
            display(HTML(f"<div style='background:#f8d7da; padding:15px; border-radius:10px; color:#721c24'><b>⚠️ Error:</b> {str(e)}<br><br>Please run cells 1-20 first to train the model.</div>"))

def on_clear(b):
    gender.value = 'Male'
    married.value = 'Yes'
    dependents.value = 0
    education.value = 'Graduate'
    self_employed.value = 'No'
    applicant_income.value = 5000
    coapplicant_income.value = 0
    loan_amount.value = 150
    loan_term.value = 360
    credit_history.value = 'Good (1)'
    property_area.value = 'Urban'
    with output:
        clear_output(wait=True)
        display(HTML("<div style='background:#d4edda; padding:20px; text-align:center; border-radius:12px'><h3 style='color:#155724'>✨ Form Cleared!</h3><p style='color:#155724'>Ready for new prediction</p></div>"))

def on_reset(b):
    on_clear(b)

predict_btn.on_click(on_predict)
clear_btn.on_click(on_clear)
reset_btn.on_click(on_reset)

# ============================================================
# DISPLAY UI
# ============================================================
model_acc = 85.0
if 'best_model_name' in globals() and best_model_name in results:
    model_acc = results[best_model_name]['Accuracy'] * 100

# Header
display(HTML(f"""
<div class="main-container">
    <div class="professional-header">
        <h1>🏦 LOAN APPROVAL PREDICTION SYSTEM</h1>
        <p>Professional Machine Learning Decision Support System</p>
        <div class="accuracy-badge">🎯 Model Accuracy: {model_acc:.1f}%</div>
    </div>
"""))

# Three columns
display(HTML('<div class="three-columns">'))

# Column 1
display(HTML('<div class="column">'))
col1 = VBox([
    widgets.HTML("<div class='section-title'>👤 PERSONAL INFORMATION</div>"),
    gender, married, dependents, education, self_employed
], layout=Layout(margin='0', padding='0', width='100%'))
card1 = widgets.VBox([col1], layout=Layout(margin='0', width='100%'))
card1.add_class('professional-card')
display(card1)
display(HTML('</div>'))

# Column 2
display(HTML('<div class="column">'))
col2 = VBox([
    widgets.HTML("<div class='section-title'>💰 FINANCIAL INFORMATION</div>"),
    applicant_income, applicant_value,
    coapplicant_income, coapp_value,
    total_display,
    loan_amount, loan_value, loan_term
], layout=Layout(margin='0', padding='0', width='100%'))
card2 = widgets.VBox([col2], layout=Layout(margin='0', width='100%'))
card2.add_class('professional-card')
display(card2)
display(HTML('</div>'))

# Column 3
display(HTML('<div class="column">'))
col3 = VBox([
    widgets.HTML("<div class='section-title'>📊 CREDIT & PROPERTY</div>"),
    credit_history, property_area
], layout=Layout(margin='0', padding='0', width='100%'))
card3 = widgets.VBox([col3], layout=Layout(margin='0', width='100%'))
card3.add_class('professional-card')
display(card3)
display(HTML('</div>'))

display(HTML('</div>'))

# Buttons
button_row = HBox([predict_btn, clear_btn, reset_btn],
                  layout=Layout(justify_content='center', gap='20px', margin='20px 0', flex_wrap='wrap'))
display(button_row)

# Output
display(HTML("<h2 style='color:#1a237e; margin: 20px 0 10px 0'>📊 PREDICTION RESULT</h2>"))
display(output)

# Tips
display(HTML("""
<div class="tips-panel">
    <h4>💡 PROFESSIONAL TIPS</h4>
    <ul>
        <li>✅ <strong>Ideal Candidate:</strong> Income > $5,000 | Good Credit | Loan < $200K → High approval</li>
        <li>❌ <strong>High Risk:</strong> Income < $3,000 | Bad Credit | Loan > $400K → Low approval</li>
        <li>⭐ <strong>Most Important:</strong> Credit History has the biggest impact on approval</li>
        <li>📈 <strong>Key Metric:</strong> Higher income-to-loan ratio = Better chances</li>
    </ul>
</div>
</div>
"""))

print("\n" + "="*60)
print("✅ UI FIXED! Dropdowns and sliders are now FULLY VISIBLE with no cut-off")
print("="*60)